In [1]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix,
                              mean_squared_error, r2_score)
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/featured.csv')

# Load all saved models
with open('../models/rf_classifier.pkl', 'rb') as f:
    rf_model = pickle.load(f)

with open('../models/xgb_classifier.pkl', 'rb') as f:
    xgb_model = pickle.load(f)

with open('../models/gbm_regressor.pkl', 'rb') as f:
    gbm_model = pickle.load(f)

print("✅ All models loaded!")

✅ All models loaded!


In [7]:
from sklearn.model_selection import train_test_split

# Classification features — same as Phase 5 (no gap_score)
clf_features = [
    'Danceability', 'Energy', 'Loudness', 'Speechiness',
    'Acousticness', 'Instrumentalness', 'Liveness', 'Valence',
    'Tempo', 'duration_min', 'engagement_ratio',
    'energy_dance', 'positivity_score',
    'artist_platform_score', 'album_type_encoded'
]

# Regression features — same as Phase 6 (includes gap_score)
reg_features = [
    'Danceability', 'Energy', 'Loudness', 'Speechiness',
    'Acousticness', 'Instrumentalness', 'Liveness', 'Valence',
    'Tempo', 'duration_min', 'engagement_ratio',
    'energy_dance', 'positivity_score',
    'artist_platform_score', 'album_type_encoded', 'gap_score'
]

X_clf = df[clf_features]
X_reg = df[reg_features]
y_class = df['platform_label']
y_reg   = df['log_streams']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_class, test_size=0.2, random_state=42, stratify=y_class)

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42)

print("✅ Data ready!")

✅ Data ready!


In [8]:
# Cross validation tests the model on 5 different splits
# More reliable than a single train/test split

rf_cv  = cross_val_score(rf_model,  X_clf, y_class, cv=5, scoring='f1_weighted')
xgb_cv = cross_val_score(xgb_model, X_clf, y_class, cv=5, scoring='f1_weighted')

print("✅ Cross Validation F1 Scores:")
print(f"Random Forest  : {rf_cv.mean():.4f} ± {rf_cv.std():.4f}")
print(f"XGBoost        : {xgb_cv.mean():.4f} ± {xgb_cv.std():.4f}")

✅ Cross Validation F1 Scores:
Random Forest  : 0.6391 ± 0.0129
XGBoost        : 0.6405 ± 0.0085


In [9]:
labels = ['YouTube-native', 'Cross-platform', 'Spotify-native']

rf_preds  = rf_model.predict(X_test_c)
xgb_preds = xgb_model.predict(X_test_c)

print("── Random Forest ───────────────────────")
print(classification_report(y_test_c, rf_preds, target_names=labels))

print("── XGBoost ─────────────────────────────")
print(classification_report(y_test_c, xgb_preds, target_names=labels))

── Random Forest ───────────────────────
                precision    recall  f1-score   support

YouTube-native       0.69      0.72      0.71      1290
Cross-platform       0.55      0.49      0.52      1290
Spotify-native       0.70      0.75      0.72      1330

      accuracy                           0.65      3910
     macro avg       0.65      0.65      0.65      3910
  weighted avg       0.65      0.65      0.65      3910

── XGBoost ─────────────────────────────
                precision    recall  f1-score   support

YouTube-native       0.70      0.72      0.71      1290
Cross-platform       0.54      0.51      0.53      1290
Spotify-native       0.70      0.72      0.71      1330

      accuracy                           0.65      3910
     macro avg       0.65      0.65      0.65      3910
  weighted avg       0.65      0.65      0.65      3910



In [11]:
gbm_preds = gbm_model.predict(X_test_r)

rmse = np.sqrt(mean_squared_error(y_test_r, gbm_preds))
r2   = r2_score(y_test_r, gbm_preds)

print("── Gradient Boosting Regressor ─────────")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

── Gradient Boosting Regressor ─────────
RMSE : 1.4137
R²   : 0.3020


In [12]:
summary = pd.DataFrame({
    'Model'     : ['Random Forest', 'XGBoost', 'Gradient Boosting'],
    'Task'      : ['Classification', 'Classification', 'Regression'],
    'Metric'    : ['F1 Score', 'F1 Score', 'R²'],
    'Score'     : [rf_cv.mean().round(4),
                   xgb_cv.mean().round(4),
                   round(r2, 4)],
    'RMSE/Std'  : [rf_cv.std().round(4),
                   xgb_cv.std().round(4),
                   round(rmse, 4)]
})

print("✅ Final Model Summary:")
print(summary.to_string(index=False))

# Save summary
summary.to_csv('../data/outputs/model_summary.csv', index=False)
print("\n✅ Saved to data/outputs/model_summary.csv")

✅ Final Model Summary:
            Model           Task   Metric  Score  RMSE/Std
    Random Forest Classification F1 Score 0.6391    0.0129
          XGBoost Classification F1 Score 0.6405    0.0085
Gradient Boosting     Regression       R² 0.3020    1.4137

✅ Saved to data/outputs/model_summary.csv
